# Setup

In [ ]:
import sys
from pathlib import Path
import polars as pl
sys.path.append(str(Path().resolve().parent))

from functions.isochrones import build_isochrone_catchments
from functions.data_loading import load_geodata
import polars as pl

# Data loading

In [ ]:
panel = pl.read_parquet(r"X:\Project-Shares\germart-lite\germart-papaio\panel")
gdf, kreis_gdf, bundesland_gdf = load_geodata()

In [ ]:
# from functions.distances import nearest_competitors
# nearest_competitors = nearest_competitors(panel, year=2024, k=500)
# nearest_competitors.write_parquet(r"X:\Project-Shares\germart-lite\germart-papaio\edeka_project\\data\nearest competitors\nearest_competitors.parquet")

In [ ]:
nearest_competitors = pl.read_parquet(r"X:\Project-Shares\germart-lite\germart-papaio\edeka_project\\data\nearest competitors\nearest_competitors.parquet")

In [ ]:
from functions.midpoint_catchments import (
    build_midpoint_catchments, overlap_stats, chain_contested_share,
)   

cells = build_midpoint_catchments(
    panel, year=2024, clip_to=gdf,
    shop_type=["Supermarkt", "Discount Geschäft"],
    tolerance_minutes=2.0,
    host="http://localhost:5000",
    max_workers=32,   # raise to 32 if osrm-routed's CPU isn't saturated
)

In [ ]:
import requests

def diagnose_failed_pairs(pairs, lats, lons, host, cache_path):
    import polars as pl
    done = {(r["i"], r["j"]) for r in pl.read_parquet(cache_path).iter_rows(named=True)}
    missing = [(int(i), int(j)) for i, j in pairs
               for a, b in [(int(i), int(j)), (int(j), int(i))]
               if (a, b) not in done]
    print(f"{len(missing)} pairs still missing")
    for a, b in missing:
        url = f"{host}/route/v1/driving/{lons[a]},{lats[a]};{lons[b]},{lats[b]}"
        r = requests.get(url, params={"overview": "false"}, timeout=30)
        data = r.json()
        print(f"  {a} -> {b}: code={data.get('code')}, msg={data.get('message', '')}")

# reconstruct pairs/lats/lons the same way build_midpoint_catchments does,
# or just re-run with a small wrapper — easiest is to add a debug hook:

In [ ]:
import polars as pl

df = panel.filter(pl.col("year") == 2024, pl.col("lat").is_not_null())
df = df.filter(pl.col("Shop Type").is_in(["Supermarkt", "Discount Geschäft"]))

stores = (
    df.group_by("TDLinx")
    .agg(pl.col("lat").first(), pl.col("lon").first(),
         pl.col("Gruppe").last(), pl.col("Shop Type").last())
    .sort("TDLinx")
)

lats = stores["lat"].to_numpy()
lons = stores["lon"].to_numpy()

In [ ]:
from functions.midpoint_catchments import diagnose_unroutable

diagnose_unroutable("midpoint_routes_4915c3916b837966.parquet", lats, lons, host="http://localhost:5000")

In [ ]:
print(lats[26453], lons[26453])
print(lats[19867], lons[19867])

In [ ]:
cells = overlap_stats(cells, clip_to=gdf)
chain_contested_share(cells)

# Calculate catchments for competitors

In [ ]:
catchments = build_isochrone_catchments(
    panel, year=2024, clip_to=gdf,
    # shop_type_groups=None,
    shop_type=["Supermarkt", "Discount Geschäft"],
    resolution_m=100,
    k_candidates=200,
    host="http://localhost:5000",
    checkpoint_path="isochrone_grocery_2024_candidates_200_resolution_100.npz",
)

catchments.to_file("isochrone_grocery_2024_candidates_200_resolution_100.gpkg", driver="GPKG")

In [ ]:
import geopandas as gpd
catchments_1000 = gpd.read_file("isochrone_grocery_2024_candidates20_resolution_1000.gpkg")

In [ ]:
from functions.plot_catchments import plot_catchments, plot_catchments_grid

# Single region — works for either build_voronoi() or build_isochrone_catchments() output
plot_catchments("Mannheim", catchments_1000, kreis_gdf=kreis_gdf)
plot_catchments("Baden-Württemberg", catchments_1000, bundesland_gdf=bundesland_gdf)

# Compare several Kreise side by side
plot_catchments_grid(
    ["Mannheim", "Heidelberg", "Karlsruhe", "Rhein-Neckar-Kreis"],
    catchments_1000, kreis_gdf=kreis_gdf, ncols=2,
)

In [ ]:
from functions.isochrones import route_nearest_competitors, attach_road_distances, consolidate_parts

route_nearest_competitors(
    pairs_path=r"X:\Project-Shares\germart-lite\germart-papaio\edeka_project\\data\nearest competitors\nearest_competitors.parquet",
    out_dir=r"X:\Project-Shares\germart-lite\germart-papaio\edeka_project\\data\nearest competitors\road_distances_parts",
    host="http://192.168.70.123:5000",
)
consolidate_parts(r"X:\Project-Shares\germart-lite\germart-papaio\edeka_project\\data\nearest competitors\road_distances_parts", r"X:\Project-Shares\germart-lite\germart-papaio\edeka_project\\data\nearest competitors\road_distances.parquet")

In [ ]:
from functions.isochrones import attach_road_distances
enriched = attach_road_distances(
    pairs_path=r"X:\Project-Shares\germart-lite\germart-papaio\edeka_project\\data\nearest competitors\nearest_competitors.parquet",
    road_path=r"X:\Project-Shares\germart-lite\germart-papaio\edeka_project\\data\nearest competitors\road_distances.parquet",
    out_path=r"X:\Project-Shares\germart-lite\germart-papaio\edeka_project\\data\nearest competitors\nearest_competitors_with_road.parquet",
)